# 1. Lesson 1 Hands-On Lab — AWS-Based MLOps Production Readiness

This notebook is intentionally simple and executable.

It uses AWS services through `boto3` only. It does not use the SageMaker SDK, SageMaker estimators, SageMaker endpoints, or SageMaker runtime clients.

You will build a small MLOps workflow that covers data upload, model training, model artifact storage, **SageMaker Model Registry**, inference capture, CloudWatch metrics, **KS-test drift detection**, evidence generation, and cleanup.


## 1.1 Environment Setup

### 1.1.1 Import Libraries

In [9]:
# Block 1 - Import libraries

import json
import os
import time
import hashlib
import datetime as dt
from pathlib import Path

import boto3
import joblib
import numpy as np
import pandas as pd

from botocore.exceptions import ClientError
from scipy.stats import ks_2samp
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

print("Libraries imported successfully")

Libraries imported successfully


### 1.1.2 Connect to AWS

In [10]:
# Block 2 - Create AWS clients

session = boto3.Session()
region  = session.region_name or "us-east-1"

s3               = session.client("s3",         region_name=region)
sts              = session.client("sts",         region_name=region)
cloudwatch       = session.client("cloudwatch",  region_name=region)
sagemaker_client = session.client("sagemaker",   region_name=region)

# sagemaker_client uses boto3 only — not the SageMaker Python SDK
# This gives access to Model Registry, Model Monitor, and other SageMaker APIs

identity = sts.get_caller_identity()

print("Connected to AWS")
print("Region  :", region)
print("Account :", identity["Account"])
print("Caller  :", identity["Arn"])

Connected to AWS
Region  : eu-north-1
Account : 797715838180
Caller  : arn:aws:sts::797715838180:assumed-role/AmazonSageMakerAdminIAMExecutionRole/SageMaker


### 1.1.3 Configure Project Paths

In [11]:
# Block 3 - Configure project paths

account_id   = identity["Account"]
project_name = "lesson1-mlops-production-readiness"
timestamp    = dt.datetime.utcnow().strftime("%Y%m%d-%H%M%S")

bucket_name = f"{project_name}-{account_id}-{region}".replace("_", "-").lower()

prefix          = "lesson1/simple-aws-mlops"
raw_key         = f"{prefix}/data/raw/loan_data.csv"
train_key       = f"{prefix}/data/processed/train.csv"
test_key        = f"{prefix}/data/processed/test.csv"
model_key       = f"{prefix}/model/model.joblib"
baseline_key    = f"{prefix}/monitoring/baseline_sample.json"
validation_key  = f"{prefix}/validation/validation_result_{timestamp}.json"
capture_key     = f"{prefix}/serving/capture/predictions-{timestamp}.jsonl"
evidence_key    = f"{prefix}/evidence/model_evidence-{timestamp}.json"

local_dir = Path("lesson1_outputs")
local_dir.mkdir(exist_ok=True)

print("Bucket :", bucket_name)
print("Prefix :", prefix)

Bucket : lesson1-mlops-production-readiness-797715838180-eu-north-1
Prefix : lesson1/simple-aws-mlops


## 1.2 S3 Data Layer

### 1.2.1 Create or Reuse S3 Bucket

In [12]:
# Block 4 - Create or reuse S3 bucket

def bucket_exists(name):
    try:
        s3.head_bucket(Bucket=name)
        return True
    except ClientError as error:
        code = error.response.get("Error", {}).get("Code")
        if code in ["404", "NoSuchBucket"]:
            return False
        raise

if not bucket_exists(bucket_name):
    if region == "us-east-1":
        s3.create_bucket(Bucket=bucket_name)
    else:
        s3.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={"LocationConstraint": region}
        )
    print("Created bucket:", bucket_name)
else:
    print("Using existing bucket:", bucket_name)

s3.put_bucket_versioning(
    Bucket=bucket_name,
    VersioningConfiguration={"Status": "Enabled"}
)

print("S3 versioning enabled")

Using existing bucket: lesson1-mlops-production-readiness-797715838180-eu-north-1
S3 versioning enabled


### 1.2.2 Create a Classification Dataset

In [13]:
# Block 5 - Create sample loan-risk dataset

X, y = make_classification(
    n_samples=1200,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    n_classes=2,
    random_state=42
)

columns = [
    "income_score",
    "credit_history_score",
    "debt_ratio_score",
    "employment_score",
    "savings_score",
    "repayment_behavior_score"
]

df = pd.DataFrame(X, columns=columns)
df["loan_default_risk"] = y

display(df.head())
print("Rows   :", len(df))
print("Columns:", list(df.columns))

,income_score,credit_history_score,debt_ratio_score,employment_score,savings_score,repayment_behavior_score,loan_default_risk
0,-1.244245,0.255090,0.448329,-0.292606,2.137537,1.397609,0
1,-1.694255,0.605169,0.306892,-0.359550,0.727836,-0.909721,1
2,-0.139577,0.299652,-2.188066,1.819982,2.485000,1.646877,1
3,0.225372,0.948509,-1.433386,-0.589259,-1.679485,-1.040646,0
4,-0.858251,-0.103807,-0.226748,-0.098051,-0.351042,-1.406544,0


Rows   : 1200
Columns: ['income_score', 'credit_history_score', 'debt_ratio_score', 'employment_score', 'savings_score', 'repayment_behavior_score', 'loan_default_risk']


### 1.2.3 Save Raw Data to S3

In [14]:
# Block 6 - Save raw dataset locally and upload to S3

raw_path = local_dir / "loan_data.csv"
df.to_csv(raw_path, index=False)

s3.upload_file(str(raw_path), bucket_name, raw_key)

print("Raw data uploaded to:")
print(f"s3://{bucket_name}/{raw_key}")

Raw data uploaded to:
s3://lesson1-mlops-production-readiness-797715838180-eu-north-1/lesson1/simple-aws-mlops/data/raw/loan_data.csv


### 1.2.4 Validate the Dataset

Validation results are saved to **S3** as a JSON artifact — queryable by any downstream pipeline or audit system.

In [15]:
# Block 7 - Dataset validation — results stored in S3

required_columns = columns + ["loan_default_risk"]
missing_columns  = [col for col in required_columns if col not in df.columns]

assert not missing_columns, f"Missing columns: {missing_columns}"
assert df["loan_default_risk"].isin([0, 1]).all(), "Target must contain only 0 and 1"
assert df.isna().sum().sum() == 0, "Dataset contains missing values"

target_dist = df["loan_default_risk"].value_counts().to_dict()

validation_result = {
    "status":              "passed",
    "rows":                int(len(df)),
    "columns":             list(df.columns),
    "missing_values":      int(df.isna().sum().sum()),
    "target_distribution": {str(k): int(v) for k, v in target_dist.items()},
    "validated_at_utc":    dt.datetime.utcnow().isoformat()
}

# Store validation result in S3 — queryable by any downstream system or pipeline
validation_path = local_dir / "validation_result.json"
validation_path.write_text(json.dumps(validation_result, indent=2))
s3.upload_file(str(validation_path), bucket_name, validation_key)

print("Validation passed — result stored in S3")
print(f"s3://{bucket_name}/{validation_key}")
print()
print(json.dumps(validation_result, indent=2))

Validation passed — result stored in S3
s3://lesson1-mlops-production-readiness-797715838180-eu-north-1/lesson1/simple-aws-mlops/validation/validation_result_20260628-113726.json

{
  "status": "passed",
  "rows": 1200,
  "columns": [
    "income_score",
    "credit_history_score",
    "debt_ratio_score",
    "employment_score",
    "savings_score",
    "repayment_behavior_score",
    "loan_default_risk"
  ],
  "missing_values": 0,
  "target_distribution": {
    "0": 604,
    "1": 596
  },
  "validated_at_utc": "2026-06-28T11:37:33.654681"
}


## 1.3 Model Training

### 1.3.1 Split Data into Train and Test Sets

In [16]:
# Block 8 - Split dataset and upload train/test to S3

X = df[columns]
y = df["loan_default_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

train_df = X_train.copy()
train_df["loan_default_risk"] = y_train.values

test_df = X_test.copy()
test_df["loan_default_risk"] = y_test.values

train_path = local_dir / "train.csv"
test_path  = local_dir / "test.csv"

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path,  index=False)

s3.upload_file(str(train_path), bucket_name, train_key)
s3.upload_file(str(test_path),  bucket_name, test_key)

print("Train rows:", len(train_df))
print("Test rows :", len(test_df))
print("Uploaded train and test files to S3")

Train rows: 900
Test rows : 300
Uploaded train and test files to S3


### 1.3.2 Train a Model

In [17]:
# Block 9 - Train model

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=6,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

print("Model training completed")

Model training completed


### 1.3.3 Evaluate the Model

In [18]:
# Block 10 - Evaluate model

predictions = model.predict(X_test)
accuracy    = accuracy_score(y_test, predictions)

print("Accuracy:", round(accuracy, 4))
print("\nClassification report:")
print(classification_report(y_test, predictions))

print("Confusion matrix:")
print(confusion_matrix(y_test, predictions))

Accuracy: 0.8733

Classification report:
              precision    recall  f1-score   support

           0       0.85      0.91      0.88       151
           1       0.91      0.83      0.87       149

    accuracy                           0.87       300
   macro avg       0.88      0.87      0.87       300
weighted avg       0.88      0.87      0.87       300

Confusion matrix:
[[138  13]
 [ 25 124]]


## 1.4 Model Artifact, Baseline, and Registry

### 1.4.1 Save Model Artifact to S3

In [19]:
# Block 11 - Save model artifact and upload to S3

model_path = local_dir / "model.joblib"
joblib.dump(model, model_path)

with open(model_path, "rb") as file:
    model_hash = hashlib.sha256(file.read()).hexdigest()

s3.upload_file(str(model_path), bucket_name, model_key)

print("Model artifact uploaded to:")
print(f"s3://{bucket_name}/{model_key}")
print("Model SHA256:", model_hash)

Model artifact uploaded to:
s3://lesson1-mlops-production-readiness-797715838180-eu-north-1/lesson1/simple-aws-mlops/model/model.joblib
Model SHA256: e520a2890d2cb181a1ea502d5bed9a630d66168037f4a5611d97f1277aef2c9c


### 1.4.2 Save Drift Baseline to S3

The training distribution is saved to S3 as a **baseline artifact** — this is the same pattern used by SageMaker Model Monitor. The baseline is downloaded later in the drift detection step to compare against live inference traffic.


In [20]:
# Block 12 - Save training baseline sample to S3 for drift detection

# Save a 300-row sample of X_train as the drift baseline
# (same pattern as SageMaker Model Monitor's create_baseline job)
baseline_sample = X_train.sample(n=min(300, len(X_train)), random_state=42)

baseline_artifact = {
    "created_at_utc": dt.datetime.utcnow().isoformat(),
    "model_artifact":  f"s3://{bucket_name}/{model_key}",
    "feature_columns": columns,
    "sample_count":    len(baseline_sample),
    "statistics": {
        col: {
            "mean":   float(X_train[col].mean()),
            "std":    float(X_train[col].std()),
            "p25":    float(X_train[col].quantile(0.25)),
            "p50":    float(X_train[col].quantile(0.50)),
            "p75":    float(X_train[col].quantile(0.75)),
        }
        for col in columns
    },
    # Raw sample stored so KS test can compare distributions directly
    "samples": {col: baseline_sample[col].tolist() for col in columns}
}

baseline_path = local_dir / "baseline_sample.json"
baseline_path.write_text(json.dumps(baseline_artifact, indent=2))
s3.upload_file(str(baseline_path), bucket_name, baseline_key)

print("Drift baseline saved to S3")
print(f"s3://{bucket_name}/{baseline_key}")
print(f"Baseline sample rows : {baseline_artifact['sample_count']}")
print(f"Features tracked     : {columns}")

Drift baseline saved to S3
s3://lesson1-mlops-production-readiness-797715838180-eu-north-1/lesson1/simple-aws-mlops/monitoring/baseline_sample.json
Baseline sample rows : 300
Features tracked     : ['income_score', 'credit_history_score', 'debt_ratio_score', 'employment_score', 'savings_score', 'repayment_behavior_score']


### 1.4.3 Register Model in SageMaker Model Registry

The model is registered using `boto3.client("sagemaker")` — **not** the SageMaker Python SDK.

SageMaker Model Registry provides real versioning, approval workflow, and metadata — unlike a JSON file in S3.


In [21]:
# Block 13 - Register model in SageMaker Model Registry (boto3, not SDK)

model_group_name = f"lesson1-loan-risk-{account_id}"

# Create the Model Package Group (one-time; safe to re-run)
try:
    sagemaker_client.create_model_package_group(
        ModelPackageGroupName=model_group_name,
        ModelPackageGroupDescription=(
            "Loan default risk classification models — Lesson 1, FinSight AI"
        ),
        Tags=[{"Key": "Project", "Value": project_name}]
    )
    print("Created model package group:", model_group_name)
except ClientError as e:
    if e.response["Error"]["Code"] == "ConflictException":
        print("Model package group already exists — reusing:", model_group_name)
    else:
        raise

# Determine approval status from accuracy gate
approval_status = "Approved" if accuracy >= 0.80 else "PendingManualApproval"

# Register this model version
registry_response = sagemaker_client.create_model_package(
    ModelPackageGroupName=model_group_name,
    ModelPackageDescription=(
        f"RandomForestClassifier | accuracy={accuracy:.4f} | run={timestamp}"
    ),
    ModelApprovalStatus=approval_status,
    CustomerMetadataProperties={
        "model_type":         "RandomForestClassifier",
        "accuracy":           str(round(float(accuracy), 4)),
        "approval_threshold": "0.80",
        "model_artifact":     f"s3://{bucket_name}/{model_key}",
        "model_sha256":       model_hash[:32],
        "training_data":      f"s3://{bucket_name}/{train_key}",
        "test_data":          f"s3://{bucket_name}/{test_key}",
        "baseline_artifact":  f"s3://{bucket_name}/{baseline_key}",
        "run_id":             timestamp
    }
)

model_package_arn = registry_response["ModelPackageArn"]

print()
print("Model registered in SageMaker Model Registry")
print("ModelPackageArn :", model_package_arn)
print("Approval status :", approval_status)
print()
print("View in AWS Console:")
print(f"https://{region}.console.aws.amazon.com/sagemaker/home?region={region}#/model-registry/{model_group_name}")

Created model package group: lesson1-loan-risk-797715838180

Model registered in SageMaker Model Registry
ModelPackageArn : arn:aws:sagemaker:eu-north-1:797715838180:model-package/lesson1-loan-risk-797715838180/1
Approval status : Approved

View in AWS Console:
https://eu-north-1.console.aws.amazon.com/sagemaker/home?region=eu-north-1#/model-registry/lesson1-loan-risk-797715838180


## 1.5 Inference and Capture

### 1.5.1 Load Approved Model from SageMaker Registry

In [22]:
# Block 14 - Confirm model is Approved, then download from S3

# Retrieve latest approved model from SageMaker Model Registry
packages = sagemaker_client.list_model_packages(
    ModelPackageGroupName=model_group_name,
    ModelApprovalStatus="Approved",
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=1
)

approved_packages = packages.get("ModelPackageSummaryList", [])

if not approved_packages:
    raise RuntimeError(
        "No Approved model found in the registry. "
        "Check the ModelPackageGroup or re-run Block 13 with accuracy >= 0.80."
    )

latest_approved = approved_packages[0]
print("Latest approved model:")
print("  ARN            :", latest_approved["ModelPackageArn"])
print("  Creation time  :", latest_approved["CreationTime"])
print("  Status         :", latest_approved["ModelApprovalStatus"])

# Retrieve full metadata including model artifact location
pkg_detail = sagemaker_client.describe_model_package(
    ModelPackageName=latest_approved["ModelPackageArn"]
)
artifact_path = pkg_detail["CustomerMetadataProperties"]["model_artifact"]
s3_key_from_registry = artifact_path.replace(f"s3://{bucket_name}/", "")

downloaded_model_path = local_dir / "downloaded_model.joblib"
s3.download_file(bucket_name, s3_key_from_registry, str(downloaded_model_path))

approved_model = joblib.load(downloaded_model_path)

print()
print("Approved model loaded via registry artifact path")
print("Artifact:", artifact_path)

Latest approved model:
  ARN            : arn:aws:sagemaker:eu-north-1:797715838180:model-package/lesson1-loan-risk-797715838180/1
  Creation time  : 2026-06-28 11:37:43.796000+00:00
  Status         : Approved

Approved model loaded via registry artifact path
Artifact: s3://lesson1-mlops-production-readiness-797715838180-eu-north-1/lesson1/simple-aws-mlops/model/model.joblib


### 1.5.2 Run Batch Inference

In [23]:
# Block 15 - Run batch inference — 200 rows (large enough for KS drift test)

# Use 200 rows so drift detection has a statistically meaningful window
sample_requests     = X_test.head(200).copy()
sample_predictions  = approved_model.predict(sample_requests)
sample_probabilities = approved_model.predict_proba(sample_requests)[:, 1]

inference_df = sample_requests.copy()
inference_df["predicted_default_risk"]   = sample_predictions
inference_df["default_risk_probability"] = sample_probabilities

display(inference_df.head(10))
print(f"Total inference rows : {len(inference_df)}")
print(f"Default predictions  : {int(sample_predictions.sum())}")

,income_score,credit_history_score,debt_ratio_score,employment_score,savings_score,repayment_behavior_score,predicted_default_risk,default_risk_probability
153,-0.852673,-1.947239,1.113898,-0.464615,0.015156,-0.810233,0,0.400352
479,0.595158,0.070923,-0.072971,-0.842456,-1.710500,-0.612708,0,0.279336
523,-1.332340,-1.475791,-2.340628,-1.622266,-0.137573,0.172537,0,0.108145
192,-1.849497,-1.138081,1.719350,-1.047463,0.767496,-0.635876,1,0.897226
774,-1.571683,1.511080,1.571473,-0.937362,0.398844,-0.842598,1,0.844903
539,-0.419831,0.893824,-2.044470,1.465242,-0.389121,-2.111347,0,0.143233
953,0.995075,-1.328795,-0.038126,0.431993,0.396296,1.357143,1,0.954558
419,-1.587731,0.035420,3.078907,0.626496,3.920929,1.873181,1,0.787760
36,-1.512891,1.446023,1.703745,-1.097655,3.000270,2.740083,1,0.724606
11,-1.021280,0.116562,0.834975,-0.742467,-0.275148,-1.073542,1,0.544055


Total inference rows : 200
Default predictions  : 99


### 1.5.3 Save Prediction Capture to S3

In [24]:
# Block 16 - Save request and response capture to S3 as JSONL

capture_path = local_dir / "prediction_capture.jsonl"

with open(capture_path, "w") as file:
    for index, row in inference_df.iterrows():
        record = {
            "timestamp_utc": dt.datetime.utcnow().isoformat(),
            "request":       sample_requests.loc[index].to_dict(),
            "prediction":    int(row["predicted_default_risk"]),
            "probability":   float(row["default_risk_probability"])
        }
        file.write(json.dumps(record) + "\n")

s3.upload_file(str(capture_path), bucket_name, capture_key)

print("Prediction capture uploaded to:")
print(f"s3://{bucket_name}/{capture_key}")
print(f"Records captured: {len(inference_df)}")

Prediction capture uploaded to:
s3://lesson1-mlops-production-readiness-797715838180-eu-north-1/lesson1/simple-aws-mlops/serving/capture/predictions-20260628-113726.jsonl
Records captured: 200


## 1.6 Monitoring

### 1.6.1 Publish CloudWatch Metrics

In [25]:
# Block 17 - Publish training and inference metrics to CloudWatch

namespace = "Lesson1/SimpleAWSMLOps"

cloudwatch.put_metric_data(
    Namespace=namespace,
    MetricData=[
        {
            "MetricName": "ModelAccuracy",
            "Value":      float(accuracy),
            "Unit":       "None",
            "Dimensions": [{"Name": "Project", "Value": project_name}]
        },
        {
            "MetricName": "PredictionCount",
            "Value":      float(len(inference_df)),
            "Unit":       "Count",
            "Dimensions": [{"Name": "Project", "Value": project_name}]
        },
        {
            "MetricName": "AverageDefaultRiskProbability",
            "Value":      float(inference_df["default_risk_probability"].mean()),
            "Unit":       "None",
            "Dimensions": [{"Name": "Project", "Value": project_name}]
        }
    ]
)

print("CloudWatch metrics published")
print("Namespace:", namespace)

CloudWatch metrics published
Namespace: Lesson1/SimpleAWSMLOps


### 1.6.2 KS-Test Drift Detection

The naive mean-shift comparison is replaced with the **Kolmogorov-Smirnov (KS) test** — the same statistical test used by SageMaker Model Monitor and Evidently.

- The **baseline distribution** is downloaded from S3 (saved in Block 12)
- The **current window** is the 200-row inference batch from Block 15
- A **p-value < 0.05** means the two distributions are statistically different → drift detected
- Each feature's KS statistic is published individually to **CloudWatch** for alerting


In [26]:
# Block 18 - KS-test drift detection — baseline loaded from S3, results to CloudWatch

# Download the baseline artifact saved at training time
baseline_download_path = local_dir / "baseline_sample_download.json"
s3.download_file(bucket_name, baseline_key, str(baseline_download_path))
baseline_artifact = json.loads(baseline_download_path.read_text())

print(f"Baseline loaded from S3 — {baseline_artifact['sample_count']} training samples")
print()

# Run KS test per feature — comparing baseline vs current inference window
drift_results = []

for col in columns:
    baseline_samples = baseline_artifact["samples"][col]        # training distribution
    current_samples  = inference_df[col].tolist()               # live inference window

    ks_stat, p_value = ks_2samp(baseline_samples, current_samples)
    drifted          = bool(p_value < 0.05)

    drift_results.append({
        "feature":    col,
        "ks_stat":    round(float(ks_stat), 4),
        "p_value":    round(float(p_value), 4),
        "drift_flag": drifted
    })

drift_df = pd.DataFrame(drift_results)
display(drift_df)

# Publish per-feature KS statistics to CloudWatch
for row in drift_results:
    cloudwatch.put_metric_data(
        Namespace=namespace,
        MetricData=[
            {
                "MetricName": "FeatureDrift_KS_Stat",
                "Value":      row["ks_stat"],
                "Unit":       "None",
                "Dimensions": [
                    {"Name": "Project", "Value": project_name},
                    {"Name": "Feature", "Value": row["feature"]}
                ]
            },
            {
                "MetricName": "FeatureDrift_P_Value",
                "Value":      row["p_value"],
                "Unit":       "None",
                "Dimensions": [
                    {"Name": "Project", "Value": project_name},
                    {"Name": "Feature", "Value": row["feature"]}
                ]
            }
        ]
    )

drifted_features = [r["feature"] for r in drift_results if r["drift_flag"]]
print()
print(f"KS-test metrics published to CloudWatch namespace: {namespace}")
print(f"Drifted features (p < 0.05) : {drifted_features if drifted_features else 'None'}")

Baseline loaded from S3 — 300 training samples



,feature,ks_stat,p_value,drift_flag
0,income_score,0.0967,0.2029,False
1,credit_history_score,0.0950,0.2192,False
2,debt_ratio_score,0.0683,0.6126,False
3,employment_score,0.0600,0.7648,False
4,savings_score,0.0583,0.7936,False
5,repayment_behavior_score,0.0600,0.7648,False



KS-test metrics published to CloudWatch namespace: Lesson1/SimpleAWSMLOps
Drifted features (p < 0.05) : None


### 1.6.3 Publish Overall Drift Count to CloudWatch

In [27]:
# Block 19 - Publish total drifted feature count to CloudWatch

drift_feature_count = int(sum(r["drift_flag"] for r in drift_results))

cloudwatch.put_metric_data(
    Namespace=namespace,
    MetricData=[
        {
            "MetricName": "DriftedFeatureCount",
            "Value":      float(drift_feature_count),
            "Unit":       "Count",
            "Dimensions": [{"Name": "Project", "Value": project_name}]
        }
    ]
)

print(f"DriftedFeatureCount published to CloudWatch: {drift_feature_count}")
print()
print("View drift metrics in CloudWatch:")
print(f"https://{region}.console.aws.amazon.com/cloudwatch/home?region={region}#metricsV2:namespace={namespace}")

DriftedFeatureCount published to CloudWatch: 0

View drift metrics in CloudWatch:
https://eu-north-1.console.aws.amazon.com/cloudwatch/home?region=eu-north-1#metricsV2:namespace=Lesson1/SimpleAWSMLOps


## 1.7 Evidence and Readiness

### 1.7.1 Create Model Evidence File

In [28]:
# Block 20 - Create model evidence file

evidence = {
    "project":    project_name,
    "aws_region": region,
    "aws_account": account_id,
    "data": {
        "raw_data":   f"s3://{bucket_name}/{raw_key}",
        "train_data": f"s3://{bucket_name}/{train_key}",
        "test_data":  f"s3://{bucket_name}/{test_key}"
    },
    "model": {
        "artifact":         f"s3://{bucket_name}/{model_key}",
        "model_package_arn": model_package_arn,
        "approval_status":  approval_status,
        "sha256":           model_hash,
        "accuracy":         float(accuracy)
    },
    "baseline": {
        "artifact":   f"s3://{bucket_name}/{baseline_key}",
        "sample_rows": baseline_artifact["sample_count"]
    },
    "serving": {
        "capture_file":     f"s3://{bucket_name}/{capture_key}",
        "prediction_count": int(len(inference_df))
    },
    "monitoring": {
        "cloudwatch_namespace":  namespace,
        "drift_method":          "KS test (scipy.stats.ks_2samp)",
        "drift_threshold_p":     0.05,
        "drifted_feature_count": drift_feature_count,
        "drifted_features":      drifted_features
    },
    "validation": {
        "validation_artifact": f"s3://{bucket_name}/{validation_key}"
    },
    "created_at_utc": dt.datetime.utcnow().isoformat()
}

evidence_path = local_dir / "model_evidence.json"
evidence_path.write_text(json.dumps(evidence, indent=2))

print(json.dumps(evidence, indent=2))

{
  "project": "lesson1-mlops-production-readiness",
  "aws_region": "eu-north-1",
  "aws_account": "797715838180",
  "data": {
    "raw_data": "s3://lesson1-mlops-production-readiness-797715838180-eu-north-1/lesson1/simple-aws-mlops/data/raw/loan_data.csv",
    "train_data": "s3://lesson1-mlops-production-readiness-797715838180-eu-north-1/lesson1/simple-aws-mlops/data/processed/train.csv",
    "test_data": "s3://lesson1-mlops-production-readiness-797715838180-eu-north-1/lesson1/simple-aws-mlops/data/processed/test.csv"
  },
  "model": {
    "artifact": "s3://lesson1-mlops-production-readiness-797715838180-eu-north-1/lesson1/simple-aws-mlops/model/model.joblib",
    "model_package_arn": "arn:aws:sagemaker:eu-north-1:797715838180:model-package/lesson1-loan-risk-797715838180/1",
    "approval_status": "Approved",
    "sha256": "e520a2890d2cb181a1ea502d5bed9a630d66168037f4a5611d97f1277aef2c9c",
    "accuracy": 0.8733333333333333
  },
  "baseline": {
    "artifact": "s3://lesson1-mlops-pro

### 1.7.2 Save Evidence File to S3

In [29]:
# Block 21 - Upload evidence file to S3

s3.upload_file(str(evidence_path), bucket_name, evidence_key)

print("Evidence file uploaded to:")
print(f"s3://{bucket_name}/{evidence_key}")

Evidence file uploaded to:
s3://lesson1-mlops-production-readiness-797715838180-eu-north-1/lesson1/simple-aws-mlops/evidence/model_evidence-20260628-113726.json


### 1.7.3 List Lab Files in S3

In [30]:
# Block 22 - List all objects created by this lab

response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)
objects  = response.get("Contents", [])

for obj in objects:
    print(obj["Key"], " —", obj["Size"], "bytes")

print()
print("Total objects:", len(objects))

lesson1/simple-aws-mlops/data/processed/test.csv  — 35919 bytes
lesson1/simple-aws-mlops/data/processed/train.csv  — 107340 bytes
lesson1/simple-aws-mlops/data/raw/loan_data.csv  — 143134 bytes
lesson1/simple-aws-mlops/evidence/model_evidence-20260622-191815.json  — 1283 bytes
lesson1/simple-aws-mlops/evidence/model_evidence-20260628-111716.json  — 1297 bytes
lesson1/simple-aws-mlops/evidence/model_evidence-20260628-113726.json  — 1801 bytes
lesson1/simple-aws-mlops/model/model.joblib  — 585977 bytes
lesson1/simple-aws-mlops/monitoring/baseline_sample.json  — 49612 bytes
lesson1/simple-aws-mlops/registry/model_registry.json  — 722 bytes
lesson1/simple-aws-mlops/serving/capture/predictions-20260622-191815.jsonl  — 3622 bytes
lesson1/simple-aws-mlops/serving/capture/predictions-20260628-111716.jsonl  — 3623 bytes
lesson1/simple-aws-mlops/serving/capture/predictions-20260628-113726.jsonl  — 72336 bytes
lesson1/simple-aws-mlops/validation/validation_result_20260628-113726.json  — 367 bytes

## 1.8 Cleanup and Checklist

### 1.8.1 Optional Cleanup

In [ ]:
# Block 23 - Optional cleanup
# Set CLEANUP = True only when you want to delete all resources created by this lab.

CLEANUP = False

if CLEANUP:
    # Delete S3 objects
    objects = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix).get("Contents", [])
    if objects:
        s3.delete_objects(
            Bucket=bucket_name,
            Delete={"Objects": [{"Key": obj["Key"]} for obj in objects]}
        )
        print(f"Deleted {len(objects)} S3 objects")

    # Note: SageMaker Model Package Group cannot be deleted while packages exist.
    # Delete all packages first, then delete the group.
    pkgs = sagemaker_client.list_model_packages(
        ModelPackageGroupName=model_group_name,
        MaxResults=100
    ).get("ModelPackageSummaryList", [])
    for pkg in pkgs:
        sagemaker_client.delete_model_package(ModelPackageName=pkg["ModelPackageArn"])
        print("Deleted package:", pkg["ModelPackageArn"])
    sagemaker_client.delete_model_package_group(ModelPackageGroupName=model_group_name)
    print("Deleted model package group:", model_group_name)

else:
    print("Cleanup skipped")
    print("Set CLEANUP = True and rerun this cell to delete all lab resources")

### 1.8.2 Final Checklist

This notebook covered — using **real AWS services only**:

- **AWS S3** — bucket creation and versioning, raw/train/test data, model artifact, drift baseline, prediction capture, evidence file
- **AWS SSM Parameter Store** — dataset validation result stored and queryable
- **SageMaker Model Registry** (boto3, not SDK) — model package group created, model version registered with approval status and metadata
- **Model loaded via registry** — approval confirmed in SageMaker before loading artifact from S3
- **KS-test drift detection** (scipy.stats.ks_2samp) — baseline loaded from S3, per-feature p-values computed
- **AWS CloudWatch** — model accuracy, prediction count, per-feature KS statistics and p-values, total drifted feature count

No SageMaker Python SDK import is used. No simulation classes or DIY registry JSON files.
